In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (BIOPEP-UWM)


This notebook curates toxicity-related peptide subsets from **BIOPEP-UWM** starting from the original Excel export. BIOPEP entries include free-text activity annotations; here we normalize those annotations and derive multiple binary datasets for downstream modeling and benchmarking.

- **Toxic effect / endpoint:** hemolytic, cytotoxic, embryotoxic, and toxic
- **Source:** BIOPEP-UWM
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset

The pipeline performs the following steps:
- **Loads the raw BIOPEP-UWM Excel file** and standardizes the main fields:
  - renames columns to a unified schema (`sequence`, `label`),
  - normalizes activity strings (strip + lowercase),
  - cleans sequences (removes artifacts like `~` and trims whitespace).
- **Derives activity-specific toxicity subsets** by filtering the annotation text:
  - **hemolytic**: matches “haemolytic/hemolytic” in the activity field,
  - **cytotoxic**: matches “Cytotoxic” (from the `Name` field in this export),
  - **toxic**: exact activity tag “toxic”,
  - **embryotoxic**: matches “embryotoxic” in the activity field.
  Each subset is stored as a table of `(sequence, label)` where `label = 1` indicates presence of that toxicity annotation.
- **Checks duplicates within each subset**:
  - unique sequences are kept,
  - duplicates with consistent labels are collapsed,
  - conflicting-label duplicates (unexpected here) are collected into an error table.
- **Builds metadata** from the project-wide Excel description sheet and appends QC statistics.
- **Exports curated subset datasets** and a JSON metadata file.

In [2]:
name_source = "BIOPEP-UWM"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df_biopep = pd.read_excel(f"{PATH_INPUT}/{name_source}/biopep-umw.xlsx")

In [4]:
df_biopep = (
    df_biopep
    .rename(columns={"Sequence": "sequence", "Activity": "label"})
    .assign(
        label=lambda d: d["label"].str.strip().str.lower(),
        sequence=lambda d: d["sequence"].str.replace("~", "", regex=False).str.strip()
    )
)

- Separate dataset by toxic effects

In [5]:
df_hemolytic = (
    df_biopep
    .assign(
        label=lambda d: d["label"]
        .str.contains(r"ha?emolytic", case=False, na=False) # Contains haemolytic and hemolytic
        .astype(int)
    )
    .loc[lambda d: d["label"] == 1, ["sequence", "label"]]
    .reset_index(drop=True)
)

df_hemolytic.shape

(63, 2)

In [6]:
df_cytotoxic = (
    df_biopep
    .assign(
        label=lambda d: d["Name"]
        .str.contains("Cytotoxic", case=False, na=False)
        .astype(int)
    )
    .loc[lambda d: d["label"] == 1, ["sequence", "label"]]
    .reset_index(drop=True)
)
df_cytotoxic.shape

(10, 2)

In [7]:
df_toxic = (
    df_biopep
    .loc[
        df_biopep["label"].str.lower().str.strip() == "toxic",
        ["sequence", "label"]
    ]
    .assign(label=1)
    .reset_index(drop=True)
)
df_toxic.shape

(13, 2)

In [8]:
df_embryotoxic = (
    df_biopep
    .assign(
        label=lambda d: d["label"]
        .str.contains("embryotoxic", case=False, na=False)
        .astype(int)
    )
    .loc[lambda d: d["label"] == 1, ["sequence", "label"]]
    .reset_index(drop=True)
)
df_embryotoxic.shape

(3, 2)

- Checking duplicates

In [9]:
df_remove_duplicated_hemolytic, df_errors_hemolytic, df_unique_hemolytic = processing_duplicated(df_hemolytic, group_seq="sequence", sort_key="label")

In [10]:
df_remove_duplicated_cytotoxic, df_errors_cytotoxic, df_unique_cytotoxic = processing_duplicated(df_cytotoxic, group_seq="sequence", sort_key="label")

In [11]:
df_remove_duplicated_toxic, df_errors_toxic, df_unique_toxic = processing_duplicated(df_toxic, group_seq="sequence", sort_key="label")

In [12]:
df_remove_duplicated_embryo, df_errors_embryo, df_unique_embryo = processing_duplicated(df_embryotoxic, group_seq="sequence", sort_key="label")

In [13]:
df_full_hemolytic = pd.concat([df_unique_hemolytic, df_remove_duplicated_hemolytic])
df_full_cytotoxic = pd.concat([df_unique_cytotoxic, df_remove_duplicated_cytotoxic])
df_full_toxic = pd.concat([df_unique_toxic, df_remove_duplicated_toxic])
df_full_embryo = pd.concat([df_remove_duplicated_embryo, df_unique_embryo])
df_full = pd.concat([df_full_hemolytic, df_full_cytotoxic, df_full_toxic, df_full_embryo])
df_errors = pd.concat([df_errors_hemolytic, df_errors_cytotoxic, df_errors_toxic, df_errors_embryo])

In [14]:
df_errors.shape

(0, 1)

- Working with metada

In [15]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [16]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df_biopep)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Database',
 'static-dynamic': 'Dynamic',
 'license': 'No information',
 'year of publication': 2019,
 'last update date': datetime.datetime(2025, 8, 27, 0, 0),
 'download date': Timestamp('2025-08-30 00:00:00'),
 'file format': 'xlsx',
 'peptide property': 'ACE inhibitor, antioxidant, multilabel, dipeptidyl peptidase IV inhibitor, antihypertensive, antimicrobial, antibacterial, celiac toxic, opioid, immunomodulating, immunogenic peptides, antithrombotic, anticancer, neuropeptides, alpha-glucosidase inhibitor, PEP-inhibitory, hemolytic, toxic, renin inhibitor, alpha-amylase inhibitor, dipeptidyl peptidase III inhibitor, binding peptides, antiamnestic, antifungal, heparin binding, γ-glutamyl, CaMKII Inhibitor, antiviral, cholestrol-lowering, HMG-CoA reductase inhibitor, Ileum contracting, antiinflammatory, mineral-binding, anxiolitic-like, CaMPDE inhibitor, cyclooxygenase-1 inhibitor, blood-brain barrier peptides, membrane -active , mitogenic, AChE inhibitor, osteoanabol

- Exporting data

In [17]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [18]:
df_full_hemolytic.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_hemolytic_dataset.csv", index=False)
df_full_embryo.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_embryotoxic_dataset.csv", index=False)
df_full_cytotoxic.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_cytotoxic_dataset.csv", index=False)
df_full_toxic.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_toxic_dataset.csv", index=False)